In [1]:
#Cell 1: Setup
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import recall_score

from src.config import CLASSES, MODELS_DIR, RESULTS_DIR, SEED
from src.dataset import get_loaders
from src.models import build_mobilenet_v3, count_parameters
from src.train import evaluate, fit


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HEALTHY = CLASSES.index("Potato___healthy")
print("Device:", device, "| healthy label =", HEALTHY)

Device: cpu | healthy label = 2


In [6]:
#Cell 2: A helper that runs any experiment in one line
def run_experiment(name, img_size=224, augment=True, unfreeze_last=0, epochs=10,
                   lr_head=1e-3, lr_backbone=1e-4):
    """Train + test one MobileNetV3 variant, save everything, return a summary."""
    set_seed()
    train_loader, val_loader, test_loader = get_loaders(
        img_size=img_size, batch_size=32, augment=augment
    )
    model = build_mobilenet_v3(
        num_classes=len(CLASSES), freeze_backbone=True, unfreeze_last=unfreeze_last
    ).to(device)

    # Two learning rates: normal for the new head, small for unfrozen backbone blocks
    head_params = [p for p in model.classifier.parameters() if p.requires_grad]
    backbone_params = [p for p in model.features.parameters() if p.requires_grad]
    param_groups = [{"params": head_params, "lr": lr_head}]
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": lr_backbone})

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(param_groups, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    start = time.time()
    history = fit(model, train_loader, val_loader, criterion, optimizer, device,
                  epochs=epochs, scheduler=scheduler, save_path=MODELS_DIR / f"{name}.pth")
    train_minutes = (time.time() - start) / 60

    _, test_acc, test_f1, preds, labels = evaluate(model, test_loader, criterion, device)
    healthy_recall = recall_score(labels, preds, average=None)[HEALTHY]

    result = {
        "experiment": name, "img_size": img_size, "augment": augment,
        "unfreeze_last": unfreeze_last, "trainable_params": count_parameters(model),
        "train_minutes": round(train_minutes, 1), "best_val_f1": max(history["val_f1"]),
        "test_accuracy": test_acc, "test_macro_f1": test_f1, "healthy_recall": healthy_recall,
    }
    with open(RESULTS_DIR / f"{name}_ablation.json", "w") as f:
        json.dump({**result, "history": history, "test_preds": preds, "test_labels": labels}, f, indent=2)

    print(f"\n{name}: test acc {test_acc:.3f} | macro F1 {test_f1:.3f} | "
          f"healthy recall {healthy_recall:.3f} | {train_minutes:.1f} min")
    return result

In [7]:
#Cell 3: Experiment B (no augmentation)
results = []

results.append(run_experiment("mnv3_no_aug", img_size=224, augment=False, unfreeze_last=0))

Epoch 01/10 | train loss 0.236 acc 0.912 | val loss 0.974 acc 0.616 f1 0.520 | 21s
Epoch 02/10 | train loss 0.065 acc 0.979 | val loss 0.431 acc 0.833 f1 0.749 | 19s
Epoch 03/10 | train loss 0.044 acc 0.984 | val loss 0.068 acc 0.981 f1 0.955 | 20s
Epoch 04/10 | train loss 0.047 acc 0.983 | val loss 0.053 acc 0.978 f1 0.957 | 32s
Epoch 05/10 | train loss 0.010 acc 0.999 | val loss 0.032 acc 0.978 f1 0.966 | 22s
Epoch 06/10 | train loss 0.017 acc 0.995 | val loss 0.030 acc 0.985 f1 0.977 | 32s
Epoch 07/10 | train loss 0.010 acc 0.998 | val loss 0.038 acc 0.978 f1 0.967 | 23s
Epoch 08/10 | train loss 0.008 acc 0.999 | val loss 0.039 acc 0.981 f1 0.980 | 25s
Epoch 09/10 | train loss 0.014 acc 0.997 | val loss 0.031 acc 0.981 f1 0.967 | 24s
Epoch 10/10 | train loss 0.010 acc 0.999 | val loss 0.030 acc 0.988 f1 0.985 | 24s
Best val macro F1: 0.985

mnv3_no_aug: test acc 0.985 | macro F1 0.956 | healthy recall 0.826 | 4.0 min


In [8]:
#Cell 4: Experiment C (lower resolution, 128×128)
results.append(run_experiment("mnv3_128px", img_size=128, augment=True, unfreeze_last=0))

Epoch 01/10 | train loss 0.271 acc 0.892 | val loss 0.504 acc 0.799 f1 0.727 | 10s
Epoch 02/10 | train loss 0.133 acc 0.952 | val loss 0.296 acc 0.885 f1 0.854 | 10s
Epoch 03/10 | train loss 0.188 acc 0.938 | val loss 0.189 acc 0.932 f1 0.901 | 13s
Epoch 04/10 | train loss 0.135 acc 0.943 | val loss 0.124 acc 0.960 f1 0.941 | 34s
Epoch 05/10 | train loss 0.093 acc 0.966 | val loss 0.114 acc 0.969 f1 0.959 | 12s
Epoch 06/10 | train loss 0.132 acc 0.950 | val loss 0.083 acc 0.985 f1 0.970 | 23s
Epoch 07/10 | train loss 0.097 acc 0.963 | val loss 0.088 acc 0.978 f1 0.966 | 23s
Epoch 08/10 | train loss 0.063 acc 0.978 | val loss 0.090 acc 0.975 f1 0.963 | 24s
Epoch 09/10 | train loss 0.081 acc 0.971 | val loss 0.087 acc 0.978 f1 0.966 | 24s
Epoch 10/10 | train loss 0.061 acc 0.977 | val loss 0.088 acc 0.978 f1 0.966 | 23s
Best val macro F1: 0.970

mnv3_128px: test acc 0.978 | macro F1 0.966 | healthy recall 0.913 | 3.3 min


In [9]:
#Cell 5: Experiment D (fine-tuning the last 3 blocks)
results.append(run_experiment("mnv3_finetune_last3", img_size=224, augment=True, unfreeze_last=3))

Epoch 01/10 | train loss 0.177 acc 0.934 | val loss 1.004 acc 0.693 f1 0.605 | 48s
Epoch 02/10 | train loss 0.053 acc 0.979 | val loss 0.083 acc 0.969 f1 0.941 | 65s
Epoch 03/10 | train loss 0.094 acc 0.976 | val loss 0.029 acc 0.985 f1 0.967 | 56s
Epoch 04/10 | train loss 0.066 acc 0.978 | val loss 0.016 acc 0.994 f1 0.990 | 61s
Epoch 05/10 | train loss 0.026 acc 0.991 | val loss 0.010 acc 0.997 f1 0.998 | 61s
Epoch 06/10 | train loss 0.031 acc 0.992 | val loss 0.007 acc 1.000 f1 1.000 | 45s
Epoch 07/10 | train loss 0.027 acc 0.993 | val loss 0.006 acc 1.000 f1 1.000 | 23s
Epoch 08/10 | train loss 0.013 acc 0.995 | val loss 0.005 acc 1.000 f1 1.000 | 24s
Epoch 09/10 | train loss 0.010 acc 0.997 | val loss 0.005 acc 1.000 f1 1.000 | 30s
Epoch 10/10 | train loss 0.009 acc 0.999 | val loss 0.006 acc 1.000 f1 1.000 | 24s
Best val macro F1: 1.000

mnv3_finetune_last3: test acc 0.991 | macro F1 0.981 | healthy recall 0.957 | 7.3 min


In [10]:
#Cell 6: Experiment E (fine-tune all 17 blocks)
results.append(run_experiment("mnv3_finetune_all", img_size=224, augment=True, unfreeze_last=17))


Epoch 01/10 | train loss 0.148 acc 0.945 | val loss 0.654 acc 0.820 f1 0.737 | 50s
Epoch 02/10 | train loss 0.023 acc 0.992 | val loss 0.044 acc 0.981 f1 0.959 | 55s
Epoch 03/10 | train loss 0.020 acc 0.993 | val loss 0.012 acc 0.991 f1 0.976 | 66s
Epoch 04/10 | train loss 0.040 acc 0.989 | val loss 0.012 acc 0.994 f1 0.984 | 55s
Epoch 05/10 | train loss 0.016 acc 0.994 | val loss 0.003 acc 1.000 f1 1.000 | 57s
Epoch 06/10 | train loss 0.026 acc 0.991 | val loss 0.001 acc 1.000 f1 1.000 | 59s
Epoch 07/10 | train loss 0.016 acc 0.995 | val loss 0.001 acc 1.000 f1 1.000 | 60s
Epoch 08/10 | train loss 0.005 acc 0.998 | val loss 0.001 acc 1.000 f1 1.000 | 67s
Epoch 09/10 | train loss 0.006 acc 0.998 | val loss 0.001 acc 1.000 f1 1.000 | 71s
Epoch 10/10 | train loss 0.002 acc 1.000 | val loss 0.001 acc 1.000 f1 1.000 | 76s
Best val macro F1: 1.000

mnv3_finetune_all: test acc 0.997 | macro F1 0.991 | healthy recall 0.957 | 10.3 min


In [11]:
#Cell 7: The final ablation table
# Run A comes from notebook 03, so load its saved results
with open(RESULTS_DIR / "mobilenet_v3_frozen_test_results.json") as f:
    a = json.load(f)
with open(RESULTS_DIR / "mobilenet_v3_frozen_history.json") as f:
    a_hist = json.load(f)

run_a = {
    "experiment": "mnv3_frozen (A)", "img_size": 224, "augment": True, "unfreeze_last": 0,
    "trainable_params": a["parameters_trainable"],
    "train_minutes": 4.9,  # from notebook 03 output (4m 56s)
    "best_val_f1": max(a_hist["val_f1"]),
    "test_accuracy": a["test_accuracy"], "test_macro_f1": a["test_macro_f1"],
    "healthy_recall": recall_score(a["test_labels"], a["test_preds"], average=None)[HEALTHY],
}

table = pd.DataFrame([run_a] + results)
table["test_errors"] = ((1 - table["test_accuracy"]) * 323).round().astype(int)
table = table.round(3)
table.to_csv(RESULTS_DIR / "ablation_summary.csv", index=False)
table

,experiment,img_size,augment,unfreeze_last,trainable_params,train_minutes,best_val_f1,test_accuracy,test_macro_f1,healthy_recall,test_errors
0,mnv3_frozen (A),224,True,0,1233923,4.9,0.987,0.981,0.968,0.913,6
1,mnv3_no_aug,224,False,0,1233923,4.0,0.985,0.985,0.956,0.826,5
2,mnv3_128px,128,True,0,1233923,3.3,0.970,0.978,0.966,0.913,7
3,mnv3_finetune_last3,224,True,3,2984163,7.3,1.000,0.991,0.981,0.957,3
4,mnv3_finetune_all,224,True,17,4205875,10.3,1.000,0.997,0.991,0.957,1
